# MNIST Neural Network: Beginner Level 1
## Procedural/Functional Approach

**Complexity Level**: Beginner  
**Architectural Style**: Procedural (no classes)  
**Focus**: Understanding the mathematics without OOP complexity

---

## The Problem: Handwritten Digit Recognition

The MNIST dataset presents a classic machine learning challenge: given an image of a handwritten digit, determine which digit (0-9) it represents.

**Inputs**: Each image is a 28×28 pixel grayscale image, which we flatten into a single list of 784 values. Each pixel value ranges from 0 (black) to 255 (white), though we typically normalise these to the range 0.0 to 1.0 by dividing by 255. So our neural network receives 784 floating-point numbers as input for each image.

**Outputs**: The network produces 10 values, one for each possible digit (0 through 9). These outputs represent the network's confidence that the input image shows each respective digit. We typically interpret these as probabilities (summing to 1.0 after applying softmax), and the digit with the highest probability becomes our prediction. For training, we compare these outputs against "one-hot" encoded labels where the correct digit has value 1.0 and all others have 0.0.

**The Task**: Build a function that maps 784 input values to 10 output values, learning the patterns that distinguish each digit through repeated exposure to labelled examples.

---

## Why This Approach?

This notebook uses **pure functions and lists** rather than classes. This approach:

- Makes the data flow completely explicit and traceable
- Requires no understanding of object-oriented programming
- Shows exactly what data structures hold the network's state
- Is closest to how you might implement this in a spreadsheet or on paper

**Trade-offs**:
- Less organised as complexity grows
- Weights and biases passed explicitly everywhere
- Harder to extend to different architectures

**Alternative**: We could use dictionaries instead of nested lists to name our weights (e.g., `weights['layer1']`), but lists keep indexing simple.

In [ ]:
# Only standard library imports
import math
import random

# Set seed for reproducibility
random.seed(42)

---

## Part 1: Activation Functions

Activation functions introduce non-linearity. Without them, stacking layers would just produce another linear transformation.

**Design Choice**: We implement these as simple functions, not classes. Each function takes a number (or list) and returns a transformed value.

In [ ]:
def sigmoid(x):
    """
    Sigmoid activation: squashes any value to range (0, 1).
    
    Formula: 1 / (1 + e^(-x))
    
    We clip x to avoid overflow with very large/small values.
    
    Alternative: Could use tanh which outputs (-1, 1) instead.
    """
    x = max(-500, min(500, x))  # Prevent overflow
    return 1.0 / (1.0 + math.exp(-x))


def sigmoid_derivative(x):
    """
    Derivative of sigmoid, used for backpropagation.
    
    Formula: sigmoid(x) * (1 - sigmoid(x))
    
    Note: This takes the raw input x, not the sigmoid output.
    Alternative: Could take sigmoid output directly: output * (1 - output)
    """
    s = sigmoid(x)
    return s * (1 - s)


def relu(x):
    """
    ReLU (Rectified Linear Unit): returns x if positive, else 0.
    
    Formula: max(0, x)
    
    Faster than sigmoid and helps avoid vanishing gradients.
    
    Alternative: Leaky ReLU returns small negative slope instead of 0.
    """
    return max(0.0, x)


def relu_derivative(x):
    """
    Derivative of ReLU: 1 if x > 0, else 0.
    
    At x=0 the derivative is technically undefined; we choose 0.
    Alternative: Some implementations use 0.5 at x=0.
    """
    return 1.0 if x > 0 else 0.0


def softmax(values):
    """
    Softmax: converts a list of values to probabilities summing to 1.
    
    Formula: e^(xi) / sum(e^(xj) for all j)
    
    We subtract max value for numerical stability (prevents overflow).
    
    Alternative: Could use log-softmax for more numerical stability.
    """
    max_val = max(values)
    exp_values = [math.exp(v - max_val) for v in values]
    total = sum(exp_values)
    return [ev / total for ev in exp_values]

In [ ]:
# Test activation functions
print("Sigmoid tests:")
print(f"  sigmoid(0) = {sigmoid(0):.4f} (expected: 0.5)")
print(f"  sigmoid(10) = {sigmoid(10):.4f} (expected: ~1.0)")
print(f"  sigmoid(-10) = {sigmoid(-10):.4f} (expected: ~0.0)")

print("\nReLU tests:")
print(f"  relu(5) = {relu(5)} (expected: 5)")
print(f"  relu(-5) = {relu(-5)} (expected: 0)")

print("\nSoftmax test:")
probs = softmax([1.0, 2.0, 3.0])
print(f"  softmax([1, 2, 3]) = {[f'{p:.3f}' for p in probs]}")
print(f"  sum = {sum(probs):.4f} (expected: 1.0)")

---

## Part 2: Network Initialisation

We represent the network as nested lists:
- `weights[layer][from_node][to_node]` - connection weights
- `biases[layer][node]` - bias for each node

**Design Choice**: Using nested lists keeps things simple but requires careful indexing. 

**Alternative**: Could use a dictionary with named layers, or a single flat list with computed indices.

In [ ]:
def create_network(layer_sizes):
    """
    Create weight and bias lists for a network with given layer sizes.
    
    Args:
        layer_sizes: List of integers, e.g., [784, 128, 10] for MNIST
    
    Returns:
        weights: List of 2D weight matrices between adjacent layers
        biases: List of bias vectors for each layer (except input)
    
    Weight initialisation: Random values in [-0.5, 0.5]
    
    Alternative initialisations:
    - Xavier: random * sqrt(2 / (fan_in + fan_out))
    - He: random * sqrt(2 / fan_in) - better for ReLU
    - All zeros (bad - symmetry problem)
    """
    weights = []
    biases = []
    
    for i in range(len(layer_sizes) - 1):
        fan_in = layer_sizes[i]
        fan_out = layer_sizes[i + 1]
        
        # Create weight matrix: fan_in rows, fan_out columns
        # weights[layer][from_node][to_node]
        layer_weights = []
        for _ in range(fan_in):
            node_weights = [random.uniform(-0.5, 0.5) for _ in range(fan_out)]
            layer_weights.append(node_weights)
        weights.append(layer_weights)
        
        # Create bias vector for receiving layer
        layer_biases = [0.0 for _ in range(fan_out)]  # Start at 0
        biases.append(layer_biases)
    
    return weights, biases


def count_parameters(weights, biases):
    """Count total trainable parameters in the network."""
    weight_count = sum(len(layer[0]) * len(layer) for layer in weights)
    bias_count = sum(len(layer) for layer in biases)
    return weight_count + bias_count

In [ ]:
# Create a small test network: 4 inputs -> 3 hidden -> 2 outputs
test_weights, test_biases = create_network([4, 3, 2])

print("Network structure [4, 3, 2]:")
print(f"  Weight matrices: {len(test_weights)}")
print(f"  Layer 0->1 weights shape: {len(test_weights[0])} x {len(test_weights[0][0])}")
print(f"  Layer 1->2 weights shape: {len(test_weights[1])} x {len(test_weights[1][0])}")
print(f"  Total parameters: {count_parameters(test_weights, test_biases)}")
print(f"    (4*3 + 3*2 = 18 weights, 3 + 2 = 5 biases, total = 23)")

---

## Part 3: Forward Pass

The forward pass computes the network's output for a given input. We process layer by layer, computing weighted sums and applying activations.

**Design Choice**: We store intermediate values (pre-activation and post-activation) for use in backpropagation.

In [ ]:
def forward_pass(inputs, weights, biases, hidden_activation='relu'):
    """
    Compute network output for given inputs.
    
    Args:
        inputs: List of input values (784 for MNIST)
        weights: Weight matrices from create_network
        biases: Bias vectors from create_network
        hidden_activation: 'relu' or 'sigmoid' for hidden layers
    
    Returns:
        outputs: Final layer outputs (probabilities after softmax)
        layer_inputs: Pre-activation values for each layer (for backprop)
        layer_outputs: Post-activation values for each layer (for backprop)
    
    Process for each layer:
    1. Compute weighted sum: sum(input[i] * weight[i][j]) + bias[j]
    2. Apply activation function
    3. Pass result to next layer
    """
    # Choose activation function
    if hidden_activation == 'relu':
        activate = relu
    else:
        activate = sigmoid
    
    # Store values for backpropagation
    layer_inputs = [inputs]  # Pre-activation (inputs to each layer)
    layer_outputs = [inputs]  # Post-activation (outputs from each layer)
    
    current = inputs
    
    for layer_idx in range(len(weights)):
        layer_w = weights[layer_idx]
        layer_b = biases[layer_idx]
        num_outputs = len(layer_b)
        
        # Compute weighted sums for this layer
        weighted_sums = []
        for j in range(num_outputs):
            total = layer_b[j]  # Start with bias
            for i in range(len(current)):
                total += current[i] * layer_w[i][j]
            weighted_sums.append(total)
        
        layer_inputs.append(weighted_sums)
        
        # Apply activation
        is_output_layer = (layer_idx == len(weights) - 1)
        
        if is_output_layer:
            # Output layer: use softmax for probabilities
            current = softmax(weighted_sums)
        else:
            # Hidden layer: use chosen activation
            current = [activate(x) for x in weighted_sums]
        
        layer_outputs.append(current)
    
    return current, layer_inputs, layer_outputs

In [ ]:
# Test forward pass
test_input = [0.5, 0.3, 0.8, 0.2]
output, layer_in, layer_out = forward_pass(test_input, test_weights, test_biases)

print(f"Input: {test_input}")
print(f"Hidden layer output: {[f'{x:.4f}' for x in layer_out[1]]}")
print(f"Final output (probabilities): {[f'{x:.4f}' for x in output]}")
print(f"Sum of probabilities: {sum(output):.4f}")
print(f"Predicted class: {output.index(max(output))}")

---

## Part 4: Loss Function

The loss function measures how wrong our predictions are. Lower loss = better predictions.

**Design Choice**: We use cross-entropy loss, which works well with softmax outputs.

In [ ]:
def cross_entropy_loss(predicted, target):
    """
    Cross-entropy loss for classification.
    
    Args:
        predicted: Network output probabilities (from softmax)
        target: One-hot encoded true label, e.g., [0,0,0,1,0,0,0,0,0,0] for digit 3
    
    Returns:
        loss: Scalar loss value (lower is better)
    
    Formula: -sum(target[i] * log(predicted[i]))
    
    Since target is one-hot, this simplifies to: -log(predicted[correct_class])
    
    Alternative: Mean squared error (MSE) - simpler but works less well for classification.
    """
    epsilon = 1e-15  # Prevent log(0)
    loss = 0.0
    for p, t in zip(predicted, target):
        if t > 0:  # Only non-zero targets contribute
            loss -= t * math.log(max(p, epsilon))
    return loss


def create_one_hot(label, num_classes=10):
    """Convert integer label to one-hot vector."""
    one_hot = [0.0] * num_classes
    one_hot[label] = 1.0
    return one_hot

In [ ]:
# Test loss function
# Perfect prediction for class 0
perfect_pred = [0.99, 0.01]
target_0 = [1.0, 0.0]
print(f"Perfect prediction loss: {cross_entropy_loss(perfect_pred, target_0):.4f}")

# Bad prediction for class 0
bad_pred = [0.1, 0.9]
print(f"Bad prediction loss: {cross_entropy_loss(bad_pred, target_0):.4f}")

# Random prediction
random_pred = [0.5, 0.5]
print(f"Random prediction loss: {cross_entropy_loss(random_pred, target_0):.4f}")

---

## Part 5: Backpropagation

Backpropagation computes how much each weight and bias contributed to the error, so we know how to adjust them.

**This is the most mathematically complex part.** The key idea is the chain rule from calculus: if changing weight w affects output o which affects loss L, then:

```
dL/dw = dL/do * do/dw
```

In [ ]:
def backward_pass(target, weights, layer_inputs, layer_outputs, hidden_activation='relu'):
    """
    Compute gradients for all weights and biases.
    
    Args:
        target: One-hot encoded true label
        weights: Network weights
        layer_inputs: Pre-activation values from forward pass
        layer_outputs: Post-activation values from forward pass
        hidden_activation: 'relu' or 'sigmoid'
    
    Returns:
        weight_grads: Gradients for each weight
        bias_grads: Gradients for each bias
    
    The gradient tells us which direction increases loss,
    so we move in the opposite direction during training.
    """
    # Choose derivative function
    if hidden_activation == 'relu':
        activation_derivative = relu_derivative
    else:
        activation_derivative = sigmoid_derivative
    
    num_layers = len(weights)
    weight_grads = []
    bias_grads = []
    
    # Initialise gradient storage with same structure as weights/biases
    for layer_idx in range(num_layers):
        w_grad = [[0.0] * len(weights[layer_idx][0]) for _ in range(len(weights[layer_idx]))]
        weight_grads.append(w_grad)
        bias_grads.append([0.0] * len(weights[layer_idx][0]))
    
    # Compute output layer error (softmax + cross-entropy simplifies nicely)
    output = layer_outputs[-1]
    deltas = [output[i] - target[i] for i in range(len(output))]
    
    # Backpropagate through layers (from output to input)
    for layer_idx in range(num_layers - 1, -1, -1):
        prev_output = layer_outputs[layer_idx]  # Input to this layer
        
        # Compute gradients for this layer's weights and biases
        for j in range(len(deltas)):
            bias_grads[layer_idx][j] = deltas[j]
            for i in range(len(prev_output)):
                weight_grads[layer_idx][i][j] = prev_output[i] * deltas[j]
        
        # Compute deltas for previous layer (if not at input)
        if layer_idx > 0:
            new_deltas = []
            layer_in = layer_inputs[layer_idx]  # Pre-activation of current layer
            
            for i in range(len(prev_output)):
                error_sum = 0.0
                for j in range(len(deltas)):
                    error_sum += deltas[j] * weights[layer_idx][i][j]
                # Multiply by activation derivative
                new_deltas.append(error_sum * activation_derivative(layer_in[i]))
            
            deltas = new_deltas
    
    return weight_grads, bias_grads

---

## Part 6: Training

Training updates weights and biases using gradient descent: move each parameter a small step in the direction that reduces loss.

**Design Choice**: We use basic stochastic gradient descent (SGD) - update after each example.

**Alternatives**:
- Mini-batch SGD: Average gradients over several examples before updating
- Momentum: Add fraction of previous update to current update
- Adam: Adaptive learning rates per parameter

In [ ]:
def update_parameters(weights, biases, weight_grads, bias_grads, learning_rate):
    """
    Apply gradient descent update to all parameters.
    
    Formula: parameter = parameter - learning_rate * gradient
    
    We subtract because gradient points toward increasing loss.
    """
    for layer_idx in range(len(weights)):
        # Update weights
        for i in range(len(weights[layer_idx])):
            for j in range(len(weights[layer_idx][i])):
                weights[layer_idx][i][j] -= learning_rate * weight_grads[layer_idx][i][j]
        
        # Update biases
        for j in range(len(biases[layer_idx])):
            biases[layer_idx][j] -= learning_rate * bias_grads[layer_idx][j]


def train_one_example(inputs, label, weights, biases, learning_rate, hidden_activation='relu'):
    """
    Train on a single example: forward pass, compute loss, backprop, update.
    
    Returns:
        loss: The loss for this example (before update)
        correct: Whether prediction was correct
    """
    # Forward pass
    output, layer_inputs, layer_outputs = forward_pass(
        inputs, weights, biases, hidden_activation
    )
    
    # Compute loss
    target = create_one_hot(label, len(output))
    loss = cross_entropy_loss(output, target)
    
    # Check if prediction is correct
    predicted = output.index(max(output))
    correct = (predicted == label)
    
    # Backward pass
    weight_grads, bias_grads = backward_pass(
        target, weights, layer_inputs, layer_outputs, hidden_activation
    )
    
    # Update parameters
    update_parameters(weights, biases, weight_grads, bias_grads, learning_rate)
    
    return loss, correct

In [ ]:
def train_epoch(data, labels, weights, biases, learning_rate, hidden_activation='relu'):
    """
    Train on all examples once (one epoch).
    
    Args:
        data: List of input vectors
        labels: List of integer labels
        weights, biases: Network parameters
        learning_rate: Step size for updates
    
    Returns:
        avg_loss: Average loss over all examples
        accuracy: Fraction of correct predictions
    """
    total_loss = 0.0
    correct_count = 0
    
    for inputs, label in zip(data, labels):
        loss, correct = train_one_example(
            inputs, label, weights, biases, learning_rate, hidden_activation
        )
        total_loss += loss
        correct_count += int(correct)
    
    avg_loss = total_loss / len(data)
    accuracy = correct_count / len(data)
    
    return avg_loss, accuracy


def evaluate(data, labels, weights, biases, hidden_activation='relu'):
    """
    Evaluate network on data without training.
    
    Returns:
        accuracy: Fraction of correct predictions
    """
    correct = 0
    for inputs, label in zip(data, labels):
        output, _, _ = forward_pass(inputs, weights, biases, hidden_activation)
        predicted = output.index(max(output))
        if predicted == label:
            correct += 1
    return correct / len(data)

---

## Part 7: Demonstration with Synthetic Data

Since we're not using external libraries, we'll create simple synthetic data to demonstrate the network learns.

We create a simple pattern: the label is determined by whether inputs sum above or below a threshold.

In [ ]:
def generate_synthetic_data(num_samples, input_size, num_classes):
    """
    Generate synthetic classification data.
    
    Pattern: Label is based on which portion of the input has highest sum.
    This creates a learnable but non-trivial pattern.
    """
    data = []
    labels = []
    
    chunk_size = input_size // num_classes
    
    for _ in range(num_samples):
        # Generate random input
        inputs = [random.random() for _ in range(input_size)]
        
        # Compute sum of each chunk
        chunk_sums = []
        for c in range(num_classes):
            start = c * chunk_size
            end = start + chunk_size
            chunk_sums.append(sum(inputs[start:end]))
        
        # Label is the chunk with highest sum
        label = chunk_sums.index(max(chunk_sums))
        
        data.append(inputs)
        labels.append(label)
    
    return data, labels


# Generate data
INPUT_SIZE = 20
NUM_CLASSES = 4
HIDDEN_SIZE = 16

train_data, train_labels = generate_synthetic_data(500, INPUT_SIZE, NUM_CLASSES)
test_data, test_labels = generate_synthetic_data(100, INPUT_SIZE, NUM_CLASSES)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Input size: {INPUT_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# Create and train network
weights, biases = create_network([INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES])

print(f"Network: {INPUT_SIZE} -> {HIDDEN_SIZE} -> {NUM_CLASSES}")
print(f"Total parameters: {count_parameters(weights, biases)}")
print()

# Training loop
EPOCHS = 20
LEARNING_RATE = 0.1

print("Training...")
for epoch in range(EPOCHS):
    loss, train_acc = train_epoch(
        train_data, train_labels, weights, biases, LEARNING_RATE, 'relu'
    )
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        test_acc = evaluate(test_data, test_labels, weights, biases, 'relu')
        print(f"Epoch {epoch+1:2d}: Loss={loss:.4f}, Train Acc={train_acc:.3f}, Test Acc={test_acc:.3f}")

print("\nTraining complete!")

In [ ]:
# Show some predictions
print("Sample predictions on test data:")
print("-" * 50)

for i in range(5):
    output, _, _ = forward_pass(test_data[i], weights, biases, 'relu')
    predicted = output.index(max(output))
    actual = test_labels[i]
    status = "Correct" if predicted == actual else "Wrong"
    
    print(f"Sample {i+1}: Predicted={predicted}, Actual={actual} ({status})")
    print(f"  Probabilities: {[f'{p:.3f}' for p in output]}")

---

## Summary

This procedural approach demonstrates all the core concepts of a neural network:

1. **Activation functions** transform weighted sums non-linearly
2. **Forward pass** computes predictions layer by layer
3. **Loss function** measures prediction quality
4. **Backpropagation** computes gradients using the chain rule
5. **Gradient descent** updates parameters to reduce loss

### Design Choices Made

| Choice | Alternative | Trade-off |
|--------|-------------|----------|
| Nested lists for weights | Dictionary with names | Simpler indexing vs. clearer naming |
| Random uniform init | Xavier/He init | Simpler vs. better convergence |
| SGD (one example at a time) | Mini-batch | Simpler vs. more stable gradients |
| ReLU for hidden layers | Sigmoid/tanh | Faster training vs. bounded outputs |
| Softmax + cross-entropy | MSE | Better for classification vs. simpler |

### For MNIST

To use this for actual MNIST:
1. Load MNIST data (784 inputs per image, labels 0-9)
2. Normalise pixel values to [0, 1]
3. Create network: `[784, 128, 64, 10]`
4. Train for ~10-20 epochs
5. Expect ~95%+ accuracy

---

**Next notebook**: See how this same logic can be organised using a single class.